# 🚀 IRIS Remote Coding Companion on Google Colab

**Hardware Specs:** 2 vCPUs • **12.7 GB RAM** • 100+ GB SSD • **$0 Free (No Credit Card Required)**

This notebook runs the **IRIS Remote Execution Companion** inside Google Colab's cloud environment, providing ample RAM and compute for Google Antigravity, git repositories, compilers, and unit tests.

### Instructions:
1. Click the **Play (▶)** button on the code cell below.
2. Copy the generated **`👉 REMOTE_COMPANION_URL`**.
3. Paste it into your IRIS environment (`REMOTE_COMPANION_URL`) and connect from **Coworkers ➔ Remote Cloud**!

In [ ]:
#@title Launch IRIS Remote Companion { vertical-output: true }
import os, sys, subprocess, threading, time, re

# 1. Configuration (Set your secret token)
COMPANION_SECRET = "iris_colab_secret_2026"
os.environ["IRIS_COMPANION_SECRET"] = COMPANION_SECRET
os.environ["ENVIRONMENT"] = "production"

# 2. Clone or update repository
print("[*] Setting up IRIS companion repository...")
if not os.path.exists("/content/IRIS"):
    subprocess.run(["git", "clone", "https://github.com/KIRTAN0902/IRIS.git", "/content/IRIS"], check=True)
else:
    subprocess.run(["git", "-C", "/content/IRIS", "pull"], check=True)

os.chdir("/content/IRIS/remote_companion")

# 3. Install requirements
print("[*] Installing companion dependencies...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

# 4. Install Cloudflare Tunnel (cloudflared)
if not os.path.exists("/usr/local/bin/cloudflared"):
    print("[*] Installing Cloudflare Tunnel (zero account required)...")
    subprocess.run(
        "wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared",
        shell=True,
        check=True
    )

# 5. Start Uvicorn Companion Server in background
print("[*] Starting IRIS FastAPI Companion on port 7860...")
server_proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "main:app", "--host", "127.0.0.1", "--port", "7860"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(3)

# 6. Start Cloudflare Tunnel and display public URL
print("[*] Establishing secure public HTTPS tunnel...")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:7860"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
)

tunnel_url = None
for line in tunnel_proc.stderr:
    if "trycloudflare.com" in line:
        m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if m:
            tunnel_url = m.group(0)
            break

if tunnel_url:
    print("\n" + "="*60)
    print("🎉 IRIS REMOTE CODING COMPANION IS LIVE IN GOOGLE COLAB!")
    print("="*60)
    print(f"\n👉 REMOTE_COMPANION_URL:  {tunnel_url}")
    print(f"👉 IRIS_COMPANION_SECRET: {COMPANION_SECRET}\n")
    print("="*60)
    print("1. Paste the URL into IRIS Core (.env or Vercel settings).")
    print("2. In IRIS Coworkers -> Chat, toggle to 'Remote Cloud'.")
    print("3. Leave this Colab tab running while working on tasks.")
    print("="*60 + "\n")
else:
    print("Could not extract tunnel URL. Please check cloudflared output.")

# Keep cell active
try:
    tunnel_proc.wait()
except KeyboardInterrupt:
    print("Shutting down companion...")
    tunnel_proc.terminate()
    server_proc.terminate()
